# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [14]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

# REVISI 1: Resolusi dinaikkan dari 224 ke 384 untuk mendeteksi tekstur layar/kertas
IMG_SIZE = 384 
# REVISI 2: Batch size diturunkan menjadi 16 agar memori GPU Kaggle tidak penuh
BATCH_SIZE = 16 

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED SPLIT
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

train_df, valid_df = train_test_split(
    df_all, test_size=0.2, stratify=df_all['label'], random_state=42
)

# ==========================================
# 3. PIPELINE AUGMENTASI (ALBUMENTATIONS)
# ==========================================
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        # REVISI 3: Proteksi jika ada gambar yang korup di folder train
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 5. INISIALISASI DATALOADER
# ==========================================
train_dataset = SpoofingDataset(train_df, transforms=train_transforms)
valid_dataset = SpoofingDataset(valid_df, transforms=valid_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Prapemrosesan Selesai!")
print(f"Total Data Latih    : {len(train_dataset)} gambar")
print(f"Total Data Validasi : {len(valid_dataset)} gambar")
print(f"Kamus Label         : {id2label}")

Prapemrosesan Selesai!
Total Data Latih    : 1321 gambar
Total Data Validasi : 331 gambar
Kamus Label         : {0: 'fake_mannequin', 1: 'fake_mask', 2: 'fake_printed', 3: 'fake_screen', 4: 'fake_unknown', 5: 'realperson'}


# Membangun Arsitektur Swin

In [15]:
import torch
import torch.nn as nn
import timm

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. INISIALISASI MODEL SWIN TRANSFORMER
# ==========================================
# Menggunakan versi 'base' dengan ukuran patch 4 dan window 12 untuk resolusi 384x384.
# Jika memori GPU penuh, pertimbangkan untuk menurunkan batch_size di Tahap 1.
MODEL_NAME = 'swin_base_patch4_window12_384.ms_in22k'
NUM_CLASSES = 6 

print(f"Mengunduh dan membangun model: {MODEL_NAME}...")

# Mengatur num_classes=6 akan otomatis menyesuaikan linear layer terakhir
model = timm.create_model(
    MODEL_NAME, 
    pretrained=True, 
    num_classes=NUM_CLASSES
)

model = model.to(device)

# ==========================================
# 3. UJI COBA MODEL (SANITY CHECK)
# ==========================================
# Dummy data sekarang disesuaikan menjadi ukuran 384x384
dummy_input = torch.randn(4, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Model berhasil dibangun!")
print(f"Bentuk input (Batch, Channel, Height, Width) : {dummy_input.shape}")
print(f"Bentuk output (Batch, Num_Classes)           : {dummy_output.shape}")

Menggunakan perangkat: cuda
Mengunduh dan membangun model: swin_base_patch4_window12_384.ms_in22k...


model.safetensors:   0%|          | 0.00/451M [00:00<?, ?B/s]

Model berhasil dibangun!
Bentuk input (Batch, Channel, Height, Width) : torch.Size([4, 3, 384, 384])
Bentuk output (Batch, Num_Classes)           : torch.Size([4, 6])


# Strategi Pelatihan (Warm-up & Loss)

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. DEFINISI FOCAL LOSS UNTUK KELAS IMBALANS
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# ==========================================
# 2. INISIALISASI OPTIMIZER, LOSS, & SCHEDULER
# ==========================================
# REVISI 1: Meningkatkan Epoch dari 10 menjadi 30
EPOCHS = 30  
LEARNING_RATE = 2e-5 
# REVISI 2: Meningkatkan batas toleransi Early Stopping
PATIENCE = 5 

criterion = FocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ==========================================
# 3. FUNGSI PELATIHAN (TRAINING LOOP)
# ==========================================
best_val_f1 = 0.0
patience_counter = 0

print("Memulai Pelatihan Swin Transformer...")

for epoch in range(EPOCHS):
    # --- FASE TRAINING ---
    model.train()
    train_loss = 0.0
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad() 
        outputs = model(images) 
        loss = criterion(outputs, labels) 
        
        loss.backward() 
        optimizer.step() 
        
        train_loss += loss.item() * images.size(0)
        train_bar.set_postfix(loss=loss.item())
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- FASE VALIDASI ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
    with torch.no_grad(): 
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(valid_loader.dataset)
    
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\nHasil Epoch {epoch+1}:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step()
    
    # --- SIMPAN MODEL TERBAIK & EARLY STOPPING ---
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), 'best_swin_model.pth')
        print("Model membaik! Disimpan ke 'best_swin_model.pth'")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"Model tidak membaik. Kesabaran: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print("\nEarly Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.")
        break

print(f"\nPelatihan Selesai! Skor Macro F1 Validasi Terbaik: {best_val_f1:.4f}")

Memulai Pelatihan Swin Transformer...


Epoch 1/30 [Valid]: 100%|██████████| 21/21 [00:14<00:00,  1.46it/s]



Hasil Epoch 1:
Train Loss: 0.5945 | Val Loss: 0.4454 | Val Macro F1: 0.7360
Model membaik! Disimpan ke 'best_swin_model.pth'


Epoch 2/30 [Valid]: 100%|██████████| 21/21 [00:14<00:00,  1.46it/s]



Hasil Epoch 2:
Train Loss: 0.2862 | Val Loss: 0.3733 | Val Macro F1: 0.8094
Model membaik! Disimpan ke 'best_swin_model.pth'


...


Hasil Epoch 5:
Train Loss: 0.1067 | Val Loss: 0.4228 | Val Macro F1: 0.7740
Model tidak membaik. Kesabaran: 3/5


Epoch 6/30 [Valid]: 100%|██████████| 21/21 [00:14<00:00,  1.46it/s]



Hasil Epoch 6:
Train Loss: 0.1071 | Val Loss: 0.4065 | Val Macro F1: 0.7695
Model tidak membaik. Kesabaran: 4/5


Epoch 7/30 [Valid]: 100%|██████████| 21/21 [00:14<00:00,  1.45it/s]


Hasil Epoch 7:
Train Loss: 0.0962 | Val Loss: 0.4429 | Val Macro F1: 0.7453
Model tidak membaik. Kesabaran: 5/5

Early Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.

Pelatihan Selesai! Skor Macro F1 Validasi Terbaik: 0.8094


# Kalibrasi Probabilitas (Thresholding)

In [17]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score
from scipy.optimize import minimize
import warnings

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

# ==========================================
# 1. MUAT MODEL TERBAIK & PERSIAPAN DATA
# ==========================================
print("Memuat bobot model terbaik...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load('best_swin_model.pth', map_location=device))
model.eval()

all_probs = []
all_labels = []

print("Mengekstrak probabilitas mentah dari Set Validasi...")
with torch.no_grad():
    for images, labels in valid_loader: # Menggunakan valid_loader dari Tahap 1
        images = images.to(device)
        outputs = model(images)
        # Ubah output logits (mentah) menjadi probabilitas 0.0 - 1.0
        probs = F.softmax(outputs, dim=1) 
        
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ==========================================
# 2. EVALUASI SEBELUM KALIBRASI
# ==========================================
preds_before = np.argmax(all_probs, axis=1)
f1_before = f1_score(all_labels, preds_before, average='macro')
print(f"Macro F1-Score SEBELUM Kalibrasi : {f1_before:.5f}")

# ==========================================
# 3. PROSES OPTIMASI THRESHOLD
# ==========================================
# Fungsi untuk mencari nilai 'F1 terburuk' (scipy mencari nilai minimum)
def optimize_f1(weights):
    # Kalikan probabilitas model dengan bobot tebakan algoritma
    weighted_probs = all_probs * weights
    # Ambil kelas dengan probabilitas tertinggi setelah dikalikan bobot
    calibrated_preds = np.argmax(weighted_probs, axis=1)
    # Kembalikan nilai negatif F1
    return -1.0 * f1_score(all_labels, calibrated_preds, average='macro')

# Inisialisasi bobot awal (semua kelas dikalikan 1.0 / tidak diubah)
initial_weights = [1.0] * 6 

print("Memulai optimasi kalibrasi probabilitas (Nelder-Mead)...")
result = minimize(optimize_f1, initial_weights, method='Nelder-Mead')

best_weights = result.x

# ==========================================
# 4. EVALUASI SETELAH KALIBRASI
# ==========================================
weighted_probs_after = all_probs * best_weights
preds_after = np.argmax(weighted_probs_after, axis=1)
f1_after = f1_score(all_labels, preds_after, average='macro')

print(f"Macro F1-Score SETELAH Kalibrasi : {f1_after:.5f}")
print(f"Peningkatan Skor                 : {f1_after - f1_before:.5f}")
print("\nSimpan array bobot ini untuk dipakai di Test Set (Tahap 5):")
print(f"best_weights = {list(best_weights)}")

Memuat bobot model terbaik...
Mengekstrak probabilitas mentah dari Set Validasi...
Macro F1-Score SEBELUM Kalibrasi : 0.80940
Memulai optimasi kalibrasi probabilitas (Nelder-Mead)...
Macro F1-Score SETELAH Kalibrasi : 0.80940
Peningkatan Skor                 : 0.00000

Simpan array bobot ini untuk dipakai di Test Set (Tahap 5):
best_weights = [np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0), np.float64(1.0)]


# Prediksi TTA & Pengumpulan (Submission)

In [19]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

# REVISI 1: Menyesuaikan Resize dengan resolusi yang baru (384x384)
IMG_SIZE = 384

test_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            # REVISI 2: Dummy image juga harus 384x384
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# ==========================================
# 3. EKSEKUSI PREDIKSI (INFERENCE)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Ganti list ini dengan hasil dari Tahap 4 (Kalibrasi Probabilitas) jika ada
best_weights = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0]) 

all_predictions = []

print(f"Memulai prediksi {len(sub_df)} data (Sistem Anti-Crash Aktif)...")

with torch.no_grad():
    for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc="Menebak Data"):
        imgs_orig = imgs_orig.to(device)
        imgs_flip = imgs_flip.to(device)

        out_orig = model(imgs_orig)
        prob_orig = F.softmax(out_orig, dim=1)

        out_flip = model(imgs_flip)
        prob_flip = F.softmax(out_flip, dim=1)

        prob_avg = (prob_orig + prob_flip) / 2.0
        prob_avg = prob_avg.cpu().numpy()

        weighted_probs = prob_avg * best_weights
        final_preds = np.argmax(weighted_probs, axis=1)

        all_predictions.extend(final_preds)

# ==========================================
# 4. PENYIMPANAN KE TEMPLATE
# ==========================================
print("Menyusun file submission...")

final_labels = [id2label[pred] for pred in all_predictions]

sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' berhasil dibuat tanpa error.")
print(sub_df.head())

Memulai prediksi 404 data (Sistem Anti-Crash Aktif)...


Menebak Data: 100%|██████████| 13/13 [00:34<00:00,  2.63s/it]

Menyusun file submission...
Selesai! File 'submission.csv' berhasil dibuat tanpa error.
         id        label
0  test_001  fake_screen
1  test_002  fake_screen
2  test_003    fake_mask
3  test_004   realperson
4  test_005    fake_mask


In [ ]:
#0,65078